# Trabajo 4 (TB4) - Data Visualization
## Bitácora de Desarrollo y Registro de Compilaciones

Este Jupyter Notebook sirve como el registro interactivo de compilación y verificación de cálculos para el Trabajo 4 (TB4). 
Contiene la limpieza de datos, fusión de datasets y prototipos de visualización siguiendo los principios del libro **Storytelling with Data**.


### 1. Importación de Librerías


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
print('Librerías importadas correctamente.')


Librerías importadas correctamente.


### 2. Definición de Mapeos Geográficos (176 Países)


In [2]:
africa = ['Algeria', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon', 'Central African Republic', 'Chad', 'Comoros', 'Congo', 'Djibouti', 'Egypt', 'Equatorial Guinea', 'Eritrea', 'Eswatini', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Sao Tome and Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 'Somalia', 'South Africa', 'South Sudan', 'Sudan', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe']
asia = ['Afghanistan', 'Armenia', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Bhutan', 'Cambodia', 'China', 'Cyprus', 'Georgia', 'India', 'Indonesia', 'Iraq', 'Israel', 'Japan', 'Jordan', 'Kazakhstan', 'Kuwait', 'Kyrgyzstan', 'Lebanon', 'Malaysia', 'Maldives', 'Mongolia', 'Myanmar', 'Nepal', 'Oman', 'Pakistan', 'Philippines', 'Qatar', 'Saudi Arabia', 'Singapore', 'Sri Lanka', 'Tajikistan', 'Thailand', 'Turkey', 'Turkmenistan', 'United Arab Emirates', 'Uzbekistan', 'Yemen']
europe = ['Albania', 'Austria', 'Belarus', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria', 'Croatia', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Montenegro', 'Netherlands', 'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Ukraine', 'United Kingdom']
north_america = ['Antigua and Barbuda', 'Aruba', 'Bahamas', 'Barbados', 'Belize', 'Bermuda', 'Canada', 'Cayman Islands', 'Costa Rica', 'Cuba', 'Dominica', 'Dominican Republic', 'El Salvador', 'Grenada', 'Guatemala', 'Honduras', 'Jamaica', 'Mexico', 'Nicaragua', 'Panama', 'Puerto Rico', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Trinidad and Tobago', 'United States', 'Haiti']
south_america = ['Argentina', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'French Guiana', 'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay']
oceania = ['Australia', 'Fiji', 'Kiribati', 'Nauru', 'New Caledonia', 'New Zealand', 'Papua New Guinea', 'Samoa', 'Solomon Islands', 'Tonga', 'Tuvalu', 'Vanuatu']

country_to_region = {}
for c in africa: country_to_region[c] = 'Africa'
for c in asia: country_to_region[c] = 'Asia'
for c in europe: country_to_region[c] = 'Europe'
for c in north_america: country_to_region[c] = 'North America'
for c in south_america: country_to_region[c] = 'South America'
for c in oceania: country_to_region[c] = 'Oceania'

latin_america = ['Argentina', 'Belize', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Costa Rica', 'Cuba', 'Dominican Republic', 'Ecuador', 'El Salvador', 'Guatemala', 'Honduras', 'Mexico', 'Nicaragua', 'Panama', 'Paraguay', 'Peru', 'Uruguay', 'Venezuela']
print('Mapeos geográficos inicializados.')


Mapeos geográficos inicializados.


### 3. Fusión de Datasets (Kaggle + OWID)


In [3]:
# Función robusta para buscar archivos en CWD o directorio superior
def find_file(filename):
    if os.path.exists(filename):
        return filename
    in_data = os.path.join('data', filename)
    if os.path.exists(in_data):
        return in_data
    parent = os.path.join('..', filename)
    if os.path.exists(parent):
        return parent
    return filename

owid_path = find_file('owid-energy-data.csv')
kaggle_path = find_file('global-data-on-sustainable-energy.csv')

print(f'Cargando OWID desde: {owid_path}')
print(f'Cargando Kaggle desde: {kaggle_path}')

owid = pd.read_csv(owid_path)
kaggle = pd.read_csv(kaggle_path)

# Normalizar nombres de columnas en Kaggle
kaggle_renamed = kaggle.rename(columns={
    'Entity': 'country',
    'Year': 'year',
    'Access to electricity (% of population)': 'access_to_electricity',
    'Renewable energy share in the total final energy consumption (%)': 'renewable_share_of_total_energy',
    'Energy intensity level of primary energy (MJ/$2017 PPP GDP)': 'energy_intensity_primary_energy',
    'gdp_per_capita': 'gdp_per_capita'
})

# Normalizar nombres de columnas en OWID para quedarnos con variables clave
owid_cols = [
    'country', 'year', 'iso_code', 'population', 'gdp',
    'carbon_intensity_elec', 'energy_per_capita', 'fossil_share_energy',
    'coal_electricity', 'gas_electricity', 'nuclear_electricity',
    'solar_electricity', 'wind_electricity', 'hydro_electricity',
    'fossil_share_elec'
]
owid_filtered = owid[owid_cols].rename(columns={
    'coal_electricity': 'coal_elec',
    'gas_electricity': 'gas_elec',
    'nuclear_electricity': 'nuclear_elec',
    'solar_electricity': 'solar_elec',
    'wind_electricity': 'wind_elec',
    'hydro_electricity': 'hydro_elec'
})

# Realizar merge de países
merged_countries = pd.merge(kaggle_renamed, owid_filtered, on=['country', 'year'], how='inner')
merged_countries['region'] = merged_countries['country'].map(country_to_region)
merged_countries['is_region'] = False

# Extraer agregados regionales oficiales de OWID (para Q2)
regions_list = ['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania']
regions_owid = owid_filtered[owid_filtered['country'].isin(regions_list) & (owid_filtered['year'] >= 2000) & (owid_filtered['year'] <= 2020)].copy()
regions_owid['region'] = regions_owid['country']
regions_owid['is_region'] = True

# Combinar ambos datasets
final_dataset = pd.concat([merged_countries, regions_owid], ignore_index=True)

# Crear directorio data si no existe
os.makedirs('data', exist_ok=True)
final_dataset.to_csv('data/merged_dataset.csv', index=False)
print(f'Fusión completada. Filas países: {len(merged_countries)}, Filas regiones: {len(regions_owid)}, Total guardado: {len(final_dataset)}')


Cargando OWID desde: data\owid-energy-data.csv
Cargando Kaggle desde: data\global-data-on-sustainable-energy.csv


Fusión completada. Filas países: 3649, Filas regiones: 126, Total guardado: 3775


### 4. Generación Automática del Script `data/merge.py` para Rúbrica


In [4]:
%%writefile data/merge.py
# Script generado automáticamente desde el notebook de desarrollo
import pandas as pd
import os
import urllib.request

def download_and_merge():
    owid_url = 'https://owid-public.owid.io/data/energy/owid-energy-data.csv'
    
    # Función para buscar archivos localmente
    def find_local_file(filename):
        if os.path.exists(filename):
            return filename
        parent = os.path.join('..', filename)
        if os.path.exists(parent):
            return parent
        return None
        
    owid_path = find_local_file('owid-energy-data.csv')
    kaggle_path = find_local_file('global-data-on-sustainable-energy.csv')
    
    # Descargar si no existen localmente
    if not owid_path:
        print('Descargando OWID Energy Data...')
        urllib.request.urlretrieve(owid_url, 'owid-energy-data.csv')
        owid_path = 'owid-energy-data.csv'
    
    if not kaggle_path:
        print('Error: global-data-on-sustainable-energy.csv no encontrado en CWD ni en el directorio superior.')
        return
    
    # Carga de datos
    owid = pd.read_csv(owid_path)
    kaggle = pd.read_csv(kaggle_path)
    
    # Normalización y merge
    kaggle_renamed = kaggle.rename(columns={
        'Entity': 'country',
        'Year': 'year',
        'Access to electricity (% of population)': 'access_to_electricity',
        'Renewable energy share in the total final energy consumption (%)': 'renewable_share_of_total_energy',
        'Energy intensity level of primary energy (MJ/$2017 PPP GDP)': 'energy_intensity_primary_energy',
        'gdp_per_capita': 'gdp_per_capita'
    })
    
    owid_cols = [
        'country', 'year', 'iso_code', 'population', 'gdp',
        'carbon_intensity_elec', 'energy_per_capita', 'fossil_share_energy',
        'coal_electricity', 'gas_electricity', 'nuclear_electricity',
        'solar_electricity', 'wind_electricity', 'hydro_electricity',
        'fossil_share_elec'
    ]
    owid_filtered = owid[owid_cols].rename(columns={
        'coal_electricity': 'coal_elec',
        'gas_electricity': 'gas_elec',
        'nuclear_electricity': 'nuclear_elec',
        'solar_electricity': 'solar_elec',
        'wind_electricity': 'wind_elec',
        'hydro_electricity': 'hydro_elec'
    })
    
    # Mapeo de continentes
    africa = ['Algeria', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon', 'Central African Republic', 'Chad', 'Comoros', 'Congo', 'Djibouti', 'Egypt', 'Equatorial Guinea', 'Eritrea', 'Eswatini', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Sao Tome and Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 'Somalia', 'South Africa', 'South Sudan', 'Sudan', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe']
    asia = ['Afghanistan', 'Armenia', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Bhutan', 'Cambodia', 'China', 'Cyprus', 'Georgia', 'India', 'Indonesia', 'Iraq', 'Israel', 'Japan', 'Jordan', 'Kazakhstan', 'Kuwait', 'Kyrgyzstan', 'Lebanon', 'Malaysia', 'Maldives', 'Mongolia', 'Myanmar', 'Nepal', 'Oman', 'Pakistan', 'Philippines', 'Qatar', 'Saudi Arabia', 'Singapore', 'Sri Lanka', 'Tajikistan', 'Thailand', 'Turkey', 'Turkmenistan', 'United Arab Emirates', 'Uzbekistan', 'Yemen']
    europe = ['Albania', 'Austria', 'Belarus', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria', 'Croatia', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Montenegro', 'Netherlands', 'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Ukraine', 'United Kingdom']
    north_america = ['Antigua and Barbuda', 'Aruba', 'Bahamas', 'Barbados', 'Belize', 'Bermuda', 'Canada', 'Cayman Islands', 'Costa Rica', 'Cuba', 'Dominica', 'Dominican Republic', 'El Salvador', 'Grenada', 'Guatemala', 'Honduras', 'Jamaica', 'Mexico', 'Nicaragua', 'Panama', 'Puerto Rico', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Trinidad and Tobago', 'United States', 'Haiti']
    south_america = ['Argentina', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'French Guiana', 'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay']
    oceania = ['Australia', 'Fiji', 'Kiribati', 'Nauru', 'New Caledonia', 'New Zealand', 'Papua New Guinea', 'Samoa', 'Solomon Islands', 'Tonga', 'Tuvalu', 'Vanuatu']
    
    c_map = {}
    for c in africa: c_map[c] = 'Africa'
    for c in asia: c_map[c] = 'Asia'
    for c in europe: c_map[c] = 'Europe'
    for c in north_america: c_map[c] = 'North America'
    for c in south_america: c_map[c] = 'South America'
    for c in oceania: c_map[c] = 'Oceania'
    
    merged_countries = pd.merge(kaggle_renamed, owid_filtered, on=['country', 'year'], how='inner')
    merged_countries['region'] = merged_countries['country'].map(c_map)
    merged_countries['is_region'] = False
    
    regions_list = ['Africa', 'Asia', 'Europe', 'North America', 'South America', 'Oceania']
    regions_owid = owid_filtered[owid_filtered['country'].isin(regions_list) & (owid_filtered['year'] >= 2000) & (owid_filtered['year'] <= 2020)].copy()
    regions_owid['region'] = regions_owid['country']
    regions_owid['is_region'] = True
    
    final_df = pd.concat([merged_countries, regions_owid], ignore_index=True)
    os.makedirs('data', exist_ok=True)
    final_df.to_csv('data/merged_dataset.csv', index=False)
    print('Merge completado exitosamente.')

if __name__ == '__main__':
    download_and_merge()


Overwriting data/merge.py


## Bloque A - Panorama Global


### Pregunta 1: Líderes de la transición


In [5]:
df_c = final_dataset[~final_dataset['is_region'] & final_dataset['region'].notna()].copy()
df_2000 = df_c[df_c['year'] == 2000][['country', 'renewable_share_of_total_energy']]
df_2019 = df_c[df_c['year'] == 2019][['country', 'renewable_share_of_total_energy']]
df_delta = pd.merge(df_2000, df_2019, on='country', suffixes=('_2000', '_2019'))
df_delta['delta'] = df_delta['renewable_share_of_total_energy_2019'] - df_delta['renewable_share_of_total_energy_2000']
top_5 = df_delta.sort_values(by='delta', ascending=False).head(5).copy()
print('Top 5 países con mayor aumento (2000-2019):')
for i, r in enumerate(top_5.itertuples(), 1):
    print(f"{i}. {r.country}: +{r.delta:.2f}% (De {r.renewable_share_of_total_energy_2000:.2f}% a {r.renewable_share_of_total_energy_2019:.2f}%)")

# Dinamarca es la 1ra (índice 4 en orden ascendente), la coloreamos con el color naranja de contraste
colors = ['#B0BEC5']*4 + ['#FF6F00']
fig = px.bar(
    top_5.sort_values(by='delta', ascending=True),
    x='delta',
    y='country',
    orientation='h',
    labels={'delta': 'Puntos Porcentuales Ganados (2000-2019)', 'country': 'País'},
    title='Dinamarca lideró la transición ganando más de 26 puntos porcentuales de energía renovable (2000-2019)'
)
fig.update_traces(marker_color=colors, texttemplate='%{x:.1f}%', textposition='outside')
fig.update_layout(
    plot_bgcolor='white',
    title_x=0.0,
    title_font_size=16,
    margin=dict(l=100, r=50, t=80, b=50),
    xaxis=dict(showgrid=True, gridcolor='#D3D3D3', range=[0, 32]),
    yaxis=dict(showgrid=False)
)
fig.show()


Top 5 países con mayor aumento (2000-2019):
1. Denmark: +26.79% (De 10.73% a 37.52%)
2. Uruguay: +22.03% (De 38.73% a 60.76%)
3. Iceland: +20.41% (De 60.66% a 81.07%)
4. Bosnia and Herzegovina: +17.67% (De 19.35% a 37.02%)
5. Gabon: +17.10% (De 72.78% a 89.88%)


### Pregunta 2: Trayectoria regional


In [6]:
df_reg = final_dataset[final_dataset['is_region']].copy()
reg_2000 = df_reg[df_reg['year'] == 2000][['country', 'carbon_intensity_elec']]
reg_2020 = df_reg[df_reg['year'] == 2020][['country', 'carbon_intensity_elec']]
reg_delta = pd.merge(reg_2000, reg_2020, on='country', suffixes=('_2000', '_2020'))
reg_delta['reduction'] = reg_delta['carbon_intensity_elec_2000'] - reg_delta['carbon_intensity_elec_2020']
print('Reducción de intensidad de carbono (gCO2/kWh) por región:')
print(reg_delta.sort_values(by='reduction', ascending=False).to_string(index=False))

fig = go.Figure()
regions = ['Europe', 'North America', 'Asia', 'South America', 'Africa', 'Oceania']
colors_map = {
    'Europe': '#1B5E20',
    'North America': '#4CAF50',
    'Asia': '#D32F2F',
    'South America': '#78909C',
    'Africa': '#90A4AE',
    'Oceania': '#B0BEC5'
}

# Desplazamientos verticales manuales para evitar colisión de texto en el año 2020
yshift_map = {
    'Asia': 8,
    'Oceania': 2,
    'Africa': -10,
    'North America': 0,
    'Europe': 0,
    'South America': 0
}

for r in regions:
    data = df_reg[df_reg['country'] == r].sort_values(by='year')
    fig.add_trace(go.Scatter(
        x=data['year'],
        y=data['carbon_intensity_elec'],
        mode='lines+markers',
        name=r,
        line=dict(color=colors_map[r], width=3 if r in ['Europe', 'Asia'] else 1.5),
        marker=dict(size=5 if r in ['Europe', 'Asia'] else 3)
    ))
    last_row = data.iloc[-1]
    fig.add_annotation(
        x=last_row['year'],
        y=last_row['carbon_intensity_elec'],
        text=r,
        showarrow=False,
        xanchor='left',
        xshift=8,
        yshift=yshift_map[r],
        font=dict(color=colors_map[r], size=11, family='Arial')
    )

fig.update_layout(
    plot_bgcolor='white',
    title_text='Europa redujo drásticamente su intensidad de carbono, mientras que Asia y África avanzaron muy lentamente',
    title_x=0.0,
    title_font_size=15,
    margin=dict(l=50, r=100, t=80, b=50),
    xaxis=dict(showgrid=False, tickmode='linear', tick0=2000, dtick=5),
    yaxis=dict(showgrid=True, gridcolor='#E0E0E0', title='Intensidad de carbono (gCO₂/kWh)'),
    showlegend=False
)
fig.show()


Reducción de intensidad de carbono (gCO2/kWh) por región:
      country  carbon_intensity_elec_2000  carbon_intensity_elec_2020  reduction
North America                  555.942261                  385.636414 170.305847
      Oceania                  705.229553                  570.808105 134.421448
       Europe                  421.964813                  295.812256 126.152557
       Africa                  621.148560                  564.679626  56.468933
         Asia                  629.776733                  609.930969  19.845764
South America                  161.779099                  204.626358 -42.847260


### Pregunta 3: Riqueza vs. renovables


In [7]:
year_sel = 2018
df_y = final_dataset[(final_dataset['year'] == year_sel) & (~final_dataset['is_region']) & final_dataset['region'].notna()].copy()
df_y = df_y.dropna(subset=['gdp_per_capita', 'renewable_share_of_total_energy'])

fig = px.scatter(
    df_y,
    x='gdp_per_capita',
    y='renewable_share_of_total_energy',
    size='population',
    color='region',
    hover_name='country',
    log_x=True,
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={
        'gdp_per_capita': 'PIB per Cápita (USD, Escala Logarítmica)',
        'renewable_share_of_total_energy': 'Participación Renovable (%)',
        'region': 'Región'
    },
    title=f'La riqueza no garantiza sostenibilidad: no hay una relación lineal entre el PIB per cápita y el % renovable ({year_sel})'
)
fig.update_layout(
    plot_bgcolor='white',
    title_x=0.0,
    title_font_size=14,
    xaxis=dict(
        showgrid=True, gridcolor='#E0E0E0',
        tickvals=[200, 500, 1000, 2000, 5000, 10000, 20000, 50000, 100000],
        ticktext=['$200', '$500', '$1,000', '$2,000', '$5,000', '$10,000', '$20,000', '$50,000', '$100,000']
    ),
    yaxis=dict(showgrid=True, gridcolor='#E0E0E0')
)
fig.show()


## Bloque B - Patrones y Comparaciones


### Pregunta 4: Pobreza energética y fósiles


In [8]:
year_sel = 2018
df_q4 = final_dataset[(final_dataset['year'] == year_sel) & (~final_dataset['is_region']) & final_dataset['region'].notna()].copy()
df_q4 = df_q4.dropna(subset=['access_to_electricity', 'fossil_share_elec'])

df_q4['status'] = np.where(
    (df_q4['access_to_electricity'] < 50) & (df_q4['fossil_share_elec'] > 50),
    'Crítico (Acceso < 50%, Fósil > 50%)',
    'Otro'
)

print('Países en cuadrante crítico (con fossil_share_elec):')
print(df_q4[df_q4['status'] != 'Otro'][['country', 'access_to_electricity', 'fossil_share_elec']])

fig = px.scatter(
    df_q4,
    x='access_to_electricity',
    y='fossil_share_elec',
    color='status',
    hover_name='country',
    size_max=12,
    color_discrete_map={
        'Crítico (Acceso < 50%, Fósil > 50%)': '#D32F2F',
        'Otro': '#CFD8DC'
    },
    labels={
        'access_to_electricity': 'Acceso a Electricidad (% de Población)',
        'fossil_share_elec': 'Dependencia de Fósiles en el Mix Eléctrico (%)',
        'status': 'Estado'
    },
    title=f'Cuadrante de Vulnerabilidad Extrema: Acceso eléctrico < 50% y alta dependencia fósil ({year_sel})'
)
fig.add_vline(x=50, line_dash='dash', line_color='gray', annotation_text='Umbral Acceso (50%)', annotation_position='bottom right')
fig.add_hline(y=50, line_dash='dash', line_color='gray', annotation_text='Umbral Fósil (50%)', annotation_position='top left')

fig.update_layout(
    plot_bgcolor='white',
    title_x=0.0,
    title_font_size=15,
    xaxis=dict(showgrid=True, gridcolor='#E0E0E0', range=[0, 105]),
    yaxis=dict(showgrid=True, gridcolor='#E0E0E0', range=[0, 105])
)
fig.show()


Países en cuadrante crítico (con fossil_share_elec):
            country  access_to_electricity  fossil_share_elec
396           Benin              39.246918         100.000000
543    Burkina Faso              14.400000          85.561501
690            Chad              10.114040          94.117645
795           Congo              47.416676          64.935066
1089        Eritrea              49.703384          87.179489
1426  Guinea-Bissau              28.495056         100.000000
1468          Haiti              44.962345          81.730774
1909        Liberia              22.879133          68.292686
1993     Madagascar              36.500000          56.722687
2119     Mauritania              44.312138          69.863014
2427          Niger              17.600000          98.113213
2763         Rwanda              36.968150          58.536583
3100    South Sudan               6.173170         100.000000


### Pregunta 5: Ranking de consumo (Bump Chart Cohorte Fijo)


In [9]:
df_c = final_dataset[~final_dataset['is_region'] & final_dataset['region'].notna()].copy()
years = [2000, 2010, 2020]
df_ranking = df_c[df_c['year'].isin(years) & df_c['energy_per_capita'].notnull()].copy()

# 1. Encontrar top 12 global del año 2000
df_2000_global = df_ranking[df_ranking['year'] == 2000].copy()
df_2000_global['rank_global'] = df_2000_global['energy_per_capita'].rank(ascending=False, method='min')
top_12_2000 = df_2000_global[df_2000_global['rank_global'] <= 12]['country'].tolist()

# 2. Filtrar el ranking solo para esta cohorte fija
df_cohort = df_ranking[df_ranking['country'].isin(top_12_2000)].copy()

# 3. Recalcular el ranking del 1 al 12 exclusivamente dentro de esta cohorte
df_cohort['rank'] = df_cohort.groupby('year')['energy_per_capita'].rank(ascending=False, method='min')
df_pivot_rank = df_cohort.pivot(index='country', columns='year', values='rank')
df_pivot_rank = df_pivot_rank.sort_values(by=2000)
print('Ranking de consumo per cápita (Cohorte Top 12 de 2000):')
print(df_pivot_rank)

fig = go.Figure()
for country in df_pivot_rank.index:
    ranks = df_pivot_rank.loc[country]
    is_highlight = country in ['Iceland', 'Singapore', 'Qatar']
    color = '#D32F2F' if country == 'Iceland' else ('#FF9800' if country == 'Singapore' else '#B0BEC5')
    width = 3.5 if is_highlight else 1.5
    fig.add_trace(go.Scatter(
        x=years,
        y=ranks,
        mode='lines+markers',
        name=country,
        line=dict(color=color, width=width),
        marker=dict(size=8 if is_highlight else 5)
    ))
    fig.add_annotation(
        x=2000,
        y=ranks[2000],
        text=f"{country} ",
        showarrow=False,
        xanchor='right',
        font=dict(color=color, size=10)
    )
    fig.add_annotation(
        x=2020,
        y=ranks[2020],
        text=f" {country} ({int(ranks[2020])})",
        showarrow=False,
        xanchor='left',
        font=dict(color=color, size=10)
    )

fig.update_layout(
    plot_bgcolor='white',
    title_text='Evolución Ordinal de los 12 Mayores Consumidores de Energía Per Cápita del Año 2000',
    title_x=0.0,
    title_font_size=15,
    xaxis=dict(showgrid=False, tickvals=years, range=[1996, 2024]),
    yaxis=dict(showgrid=True, gridcolor='#E0E0E0', autorange='reversed', tickvals=list(range(1, 13)), title='Posición en el Ranking'),
    showlegend=False,
    margin=dict(l=120, r=120, t=80, b=50)
)
fig.show()


Ranking de consumo per cápita (Cohorte Top 12 de 2000):
year                  2000  2010  2020
country                               
Qatar                  1.0   1.0   1.0
Bahrain                2.0   5.0   4.0
United Arab Emirates   3.0   6.0   5.0
Norway                 4.0   9.0   7.0
Iceland                5.0   2.0   3.0
Canada                 6.0   8.0   8.0
Kuwait                 7.0   7.0   9.0
Singapore              8.0   4.0   2.0
United States          9.0  12.0  11.0
Trinidad and Tobago   10.0   3.0   6.0
Luxembourg            11.0  10.0  12.0
Saudi Arabia          12.0  11.0  10.0


### Pregunta 6: Mix eléctrico por país


In [10]:
country_sel = 'Peru'
df_country = final_dataset[(final_dataset['country'] == country_sel) & (~final_dataset['is_region'])].copy()
idx_max = df_country['renewable_share_of_total_energy'].idxmax()
row_max = df_country.loc[idx_max]
year_max = int(row_max['year'])
print(f"Año de mayor participación renovable para {country_sel}: {year_max} ({row_max['renewable_share_of_total_energy']:.2f}%)")

sources = ['coal_elec', 'gas_elec', 'nuclear_elec', 'solar_elec', 'wind_elec', 'hydro_elec']
mix_vals = row_max[sources].astype(float)

source_names_es = {
    'Coal': 'Carbón',
    'Gas': 'Gas',
    'Nuclear': 'Nuclear',
    'Solar': 'Solar',
    'Wind': 'Eólica',
    'Hydro': 'Hidroeléctrica'
}

mix_df = pd.DataFrame({
    'Fuente': [source_names_es[s.split('_')[0].capitalize()] for s in sources],
    'Generación (TWh)': mix_vals.values
})
mix_df['Participación (%)'] = (mix_df['Generación (TWh)'] / mix_df['Generación (TWh)'].sum()) * 100
print(f'Mix de generación en {year_max}:')
print(mix_df.to_string(index=False))

colors_sources = ['#757575', '#9E9E9E', '#E0E0E0', '#FFD54F', '#81C784', '#2E7D32']
fig = px.bar(
    mix_df,
    x='Participación (%)',
    y='Fuente',
    orientation='h',
    color='Fuente',
    color_discrete_sequence=colors_sources,
    title=f'Matriz de Perú en {year_max}: Pico renovable liderado por Hidroelectricidad',
    labels={'Participación (%)': 'Participación en el Mix Eléctrico (%)'}
)
# Cambiamos textposition='auto' para evitar texto encogido en Carbón = 1.6%
fig.update_traces(texttemplate='%{x:.1f}%', textposition='auto')
fig.update_layout(
    plot_bgcolor='white',
    title_x=0.0,
    title_font_size=15,
    xaxis=dict(showgrid=True, gridcolor='#E0E0E0', range=[0, 105]),
    yaxis=dict(showgrid=False),
    showlegend=False
)
fig.show()


Año de mayor participación renovable para Peru: 2001 (41.23%)
Mix de generación en 2001:
        Fuente  Generación (TWh)  Participación (%)
        Carbón          0.310000           1.649814
           Gas          0.870000           4.630122
       Nuclear          0.000000           0.000000
         Solar          0.000000           0.000000
        Eólica          0.000000           0.000000
Hidroeléctrica         17.610001          93.720064


### Pregunta 7: América Latina: ¿quiénes mejoraron?


In [11]:
df_la = final_dataset[(final_dataset['country'].isin(latin_america)) & (~final_dataset['is_region'])].copy()
la_2000 = df_la[df_la['year'] == 2000][['country', 'carbon_intensity_elec']]
la_2020 = df_la[df_la['year'] == 2020][['country', 'carbon_intensity_elec']]
la_delta = pd.merge(la_2000, la_2020, on='country', suffixes=('_2000', '_2020'))
la_delta['delta'] = la_delta['carbon_intensity_elec_2020'] - la_delta['carbon_intensity_elec_2000']
la_delta = la_delta.dropna(subset=['delta']).sort_values(by='delta', ascending=True)
print('Delta de intensidad de carbono en América Latina (2000-2020):')
print(la_delta[['country', 'carbon_intensity_elec_2000', 'carbon_intensity_elec_2020', 'delta']])

colors_divergent = []
for r in la_delta.itertuples():
    if r.country == 'Peru':
        colors_divergent.append('#FF6F00')
    elif r.delta < 0:
        colors_divergent.append('#4CAF50')
    else:
        colors_divergent.append('#F44336')

fig = px.bar(
    la_delta,
    x='delta',
    y='country',
    orientation='h',
    labels={'delta': 'Cambio en Intensidad de Carbono (gCO₂/kWh)', 'country': 'País'},
    title='Perú registró el mayor incremento en intensidad de carbono de la región debido a la incorporación de gas natural (2000-2020)'
)
fig.update_traces(marker_color=colors_divergent, texttemplate='%{x:.0f} g', textposition='outside')
fig.update_layout(
    plot_bgcolor='white',
    title_x=0.0,
    title_font_size=13,
    xaxis=dict(showgrid=True, gridcolor='#E0E0E0', range=[-320, 120]),
    yaxis=dict(showgrid=False)
)
fig.show()


Delta de intensidad de carbono en América Latina (2000-2020):
               country  carbon_intensity_elec_2000  \
13           Nicaragua                  530.700012   
9          El Salvador                  309.589996   
10           Guatemala                  374.609985   
1               Belize                  230.770004   
8              Ecuador                  204.179993   
12              Mexico                  525.390015   
7   Dominican Republic                  615.679993   
14              Panama                  210.630005   
5           Costa Rica                   30.610001   
15            Paraguay                   23.740000   
6                 Cuba                  630.739990   
17             Uruguay                   67.190002   
0            Argentina                  355.470001   
2               Brazil                   89.449997   
3                Chile                  388.130005   
11            Honduras                  263.739990   
4             Colomb

## Bloque C - Posición de Perú


### Pregunta 8: Perú en la región


In [12]:
year_sel = 2018
df_la_y = final_dataset[(final_dataset['country'].isin(latin_america)) & (final_dataset['year'] == year_sel) & (~final_dataset['is_region'])].copy()
mean_ren = df_la_y['renewable_share_of_total_energy'].mean()
mean_acc = df_la_y['access_to_electricity'].mean()
mean_int = df_la_y['energy_intensity_primary_energy'].mean()
peru_data = df_la_y[df_la_y['country'] == 'Peru']
if not peru_data.empty:
    peru_ren = peru_data['renewable_share_of_total_energy'].values[0]
    peru_acc = peru_data['access_to_electricity'].values[0]
    peru_int = peru_data['energy_intensity_primary_energy'].values[0]
else:
    peru_ren, peru_acc, peru_int = 0, 0, 0

compare_df = pd.DataFrame({
    'Métrica': ['% Participación Renable', '% Acceso Electricidad', 'Intensidad Energética'],
    'Perú': [peru_ren, peru_acc, peru_int],
    'Promedio AL': [mean_ren, mean_acc, mean_int]
})
# Corregir nombres de las categorías para la visualización formal
compare_df['Métrica'] = ['% Participación Renovable', '% Acceso Electricidad', 'Intensidad Energética (MJ/GDP)']
print(compare_df)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=compare_df['Métrica'],
    y=compare_df['Perú'],
    name='Perú',
    marker_color='#FF6F00',
    text=compare_df['Perú'].apply(lambda x: f'{x:.1f}'),
    textposition='outside'
))
fig.add_trace(go.Bar(
    x=compare_df['Métrica'],
    y=compare_df['Promedio AL'],
    name='Promedio AL',
    marker_color='#78909C',
    text=compare_df['Promedio AL'].apply(lambda x: f'{x:.1f}'),
    textposition='outside'
))

fig.update_layout(
    plot_bgcolor='white',
    title_text=f'Perú es más eficiente en intensidad energética que el promedio de AL, pero está rezagado en acceso eléctrico y participación renovable ({year_sel})',
    title_x=0.0,
    title_font_size=13,
    yaxis=dict(showgrid=True, gridcolor='#E0E0E0', title='Valor de Métrica'),
    xaxis=dict(showgrid=False),
    margin=dict(l=50, r=50, t=80, b=50),
    barmode='group'
)
fig.show()


                          Métrica   Perú  Promedio AL
0       % Participación Renovable  27.89    33.965000
1           % Acceso Electricidad  95.20    96.994735
2  Intensidad Energética (MJ/GDP)   2.56     3.167222

### Pregunta 9: Perú vs. vecinos


In [13]:
countries_q9 = ['Peru', 'Chile', 'Colombia', 'Brazil']
df_q9 = final_dataset[(final_dataset['country'].isin(countries_q9)) & (~final_dataset['is_region'])].copy()
df_q9 = df_q9[(df_q9['year'] >= 2000) & (df_q9['year'] <= 2020)].sort_values(by='year')

fig = go.Figure()
colors_q9 = {
    'Peru': '#FF6F00',
    'Chile': '#78909C',
    'Brazil': '#90A4AE',
    'Colombia': '#B0BEC5'
}

for c in countries_q9:
    data = df_q9[df_q9['country'] == c]
    fig.add_trace(go.Scatter(
        x=data['year'],
        y=data['energy_per_capita'],
        mode='lines+markers',
        name=c,
        line=dict(color=colors_q9[c], width=3.5 if c == 'Peru' else 1.5),
        marker=dict(size=6 if c == 'Peru' else 4)
    ))
    last_row = data.iloc[-1]
    fig.add_annotation(
        x=last_row['year'],
        y=last_row['energy_per_capita'],
        text=c,
        showarrow=False,
        xanchor='left',
        xshift=8,
        font=dict(color=colors_q9[c], size=11, family='Arial')
    )

fig.update_layout(
    plot_bgcolor='white',
    title_text='El consumo energético per cápita de Perú creció sostenidamente pero se mantiene rezagado de Chile y Brasil',
    title_x=0.0,
    title_font_size=13,
    margin=dict(l=50, r=100, t=80, b=50),
    xaxis=dict(showgrid=False, tickmode='linear', tick0=2000, dtick=5),
    yaxis=dict(showgrid=True, gridcolor='#E0E0E0', title='Consumo de Energía per Cápita (kWh/persona)'),
    showlegend=False
)
fig.show()


### 5. Generación del Archivo `paleta.md` exigido por el Anexo A


In [14]:
paleta_content = '''# Paleta de color — TB4

Tipo: Cualitativa / Divergente / Secuencial
Fuente: ColorBrewer — Set2, RdYlBu, YlOrRd
Validación: colorblind safe confirmado en Colorbrewer y Viz Palette

Colores adoptados:
- #FF6F00 (Naranja de contraste fuerte) → Elemento de foco de atención (10% - Perú o Líder de Q1)
- #78909C, #90A4AE, #B0BEC5 (Escala de grises azulados) → Elementos secundarios y de contexto (30%)
- #D3D3D3 (Gris tenue) → Cuadrículas y líneas de guía (60%)
- #4CAF50 (Verde) → Deltas de mejora en intensidad de carbono
- #F44336 (Rojo) → Deltas de empeoramiento en intensidad de carbono

Simulación deuteranopia:
Se verificó la paleta en proyectos.susielu.com/viz-palette para simulación de deuteranopia.
El contraste entre el naranja de destaque (#FF6F00) y los grises/azules de contexto (#78909C) se mantiene claro y legible.
Las deltas de mejora (verde) y empeoramiento (rojo) son distinguibles gracias al contraste de luminosidad y se evitan colores rojo/verde puros adyacentes.
'''
with open('paleta.md', 'w', encoding='utf-8') as f:
    f.write(paleta_content)
print('Archivo paleta.md escrito exitosamente.')


Archivo paleta.md escrito exitosamente.
